# 02 CIGNN EMNIST

> EMNIST letter exp

In [1]:
#| default_exp data

%load_ext autoreload
%autoreload 2

In [2]:

import os
import pandas as pd
import sys
sys.path.append(os.path.abspath('..'))

import torch
import torchvision
from torchvision.transforms import ToTensor
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_image, ImageReadMode
import torch.nn as nn
import torch.optim as optim
from torchsummary import summary


/data/thallybu/data/.venv/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

n_epochs = 5
train_split = "/data/datasets/HWR/unipen_curated/split/trn.txt"
val_split = "/data/datasets/HWR/unipen_curated/split/val.txt"
test_split = "/data/datasets/HWR/unipen_curated/split/tst.txt"
data_dir = "/data/datasets/HWR/unipen_curated/curated"
model_save_path = "vgg_unipen_curated.pth"



In [4]:
# Parameters
data_dir = "/data/datasets/unipen_curated/curated"
train_split = "/data/datasets/unipen_curated/split/trn.txt"
val_split = "/data/datasets/unipen_curated/split/val.txt"
test_split = "/data/datasets/unipen_curated/split/tst.txt"


In [5]:
def unipen_char_to_code(char: str) -> int: 
    """Convert a single char into the corresponding 0-based index.

    This method respects the fact that UNIPEN (curated) does not have 
    samples for the "\"-symbol. A simple ofsetting -33 for the ASCII
    chars therefore leads to one "extra class", causing indexing errors
    down the line.

    In particular: 
        chars = [chr(i) for i in range(33, 123) if not i == 92]
        codes = [unipen_char_to_code(c) for c in chars]
        chars2 = [unipen_code_to_char(cc) for cc in codes]
        chars == chars2


    Args:
        char (str): The Char "!" - "z" to index.

    Returns:
        int: 0-based index for the input char, skipping the ascii code 92.
    """

    print(char)
    res = ord(char) - 33 
    return res - 1 if (res > (92 - 33)) else res 

def unipen_code_to_char(code: int) -> str: 
    """Convert a 0-based index int to the corresponding ASCII char in the context of the unipen curated dataset.

    This method avoids indexing errors that can happen because the unipen dataset does not provide samples for ascii char 
    92, so simple +-33 conversions cause one empty extra index.

    In particular: 
        chars = [chr(i) for i in range(33, 123) if not i == 92]
        codes = [unipen_char_to_code(c) for c in chars]
        chars2 = [unipen_code_to_char(cc) for cc in codes]
        chars == chars2

    Args:
        code (int): integer of the index to convert to char.

    Returns:
        str: The char after conversion. 
    """

    code = code if code < (92 - 33) else code + 1
    return chr(33 + code)




In [6]:

# load data from curated dataset 
class UnipenCuratedDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        self.img_labels = []
        with open(annotations_file, "r") as fh:
            self._img_labels = [[line.strip(), line.strip().split("/")[0]] for line in fh.readlines()]
            self.img_labels = pd.DataFrame(self._img_labels)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = read_image(img_path, mode=ImageReadMode.RGB)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image.to(torch.float32).to(self.device), int(label) - 33 if int(label) <= 92 else int(label) - 34


In [7]:
# Load the EMNIST dataset (one test and one training set, using the "balanced" or "letters" split for example)
train_data = UnipenCuratedDataset(train_split, data_dir)
train_dataloader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
val_data = UnipenCuratedDataset(val_split, data_dir)
val_dataloader = DataLoader(dataset=train_data, batch_size=16, shuffle=True)
test_data = UnipenCuratedDataset(test_split, data_dir)
test_dataloader = DataLoader(dataset=test_data, batch_size=16, shuffle=True)

In [8]:


# the correct #numclass is 93, however, the curated unipen dataset does not have samples for ascii 92 ("\"-symbol")
vgg_bn = torchvision.models.vgg16_bn(num_classes=93)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vgg_bn.to(device)
sample, labels = next(iter(test_dataloader))


In [9]:
pred = vgg_bn(sample)
pred.size()


torch.Size([16, 93])

In [10]:
pred = vgg_bn(sample)
labels = labels.to(device)
predicted = pred.argmax(dim=1)
print(pred.argmax(dim=1))

correct = (predicted == labels).sum().item()
correct

tensor([39, 73, 91, 68, 74, 88, 43,  4, 73, 76, 37, 91, 52, 17,  4, 74])


0

In [11]:
# Now also try the Google implementation of LeNet 
model = vgg_bn
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 1  # Define the number of epochs
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_dataloader:
        labels = labels.to(device)
        optimizer.zero_grad()  # Clear gradients
        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Calculate loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update weights
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_dataloader):.4f}")

# Testing loop
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_dataloader:
        labels = labels.to(device)
        predicted = model(images).argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

Epoch [1/1], Loss: 4.3852


Test Accuracy: 4.94%


In [12]:
torch.save(model.state_dict(), model_save_path)

# To load the saved model
# model.load_state_dict(torch.load(model_save_path))
# model.eval()  # Set the model to evaluation mode

In [13]:
# import matplotlib.pyplot as plt
# import torch

# # Function to display an image and its labels
# def show_sample(image, true_label, predicted_label):

#     # The .permute call is required to transpose the image. We receive a shape (3, 64, 64) here
#     # but .imshow() expects a shape with the channels at the end, like (64, 64, 3)
#     image = image.permute(1, 2, 0)
#     image = image.cpu().squeeze().numpy()  # Convert to 2D array for display
#     plt.imshow(image)
#     print(true_label)
#     print(predicted_label)
#     plt.title(f"True: {unipen_code_to_char(true_label)}, Predicted: {unipen_code_to_char(predicted_label)}")  
#     plt.axis('off')
#     plt.show()

# # Run inference and display samples
# model.eval()
# num_samples_to_show = 5
# shown_samples = 0

# with torch.no_grad():
#     for images, labels in test_dataloader:
#         outputs = model(images)
#         _, predicted = torch.max(outputs, 1)

#         for i in range(len(images)):
#             if shown_samples >= num_samples_to_show:
#                 break
#             true_label = labels[i].item()
#             predicted_label = predicted[i].item()
#             # Shift by -1 to match a=1, ..., z=26 in EMNIST "letters"
#             show_sample(images[i], true_label, predicted_label )
#             shown_samples += 1
        
#         if shown_samples >= num_samples_to_show:
#             break

In [14]:
#| hide
import nbdev; nbdev.nbdev_export()